# cloud

> `sysone run train.py --on kaggle|colab`: a thin adapter over the Kaggle and Colab CLIs that packages a training script, submits it, polls it, and gets the export back.

In [ ]:
#| default_exp cloud

In [ ]:
#| export
import glob, json, os, re, shlex, shutil, subprocess, tempfile, textwrap, time, uuid
from dataclasses import dataclass, field, asdict
from pathlib import Path

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail
import tempfile

Full fine-tuning goes to rented GPUs; the laptop path is head-only. `sysone.cloud` does not schedule, retry or manage infrastructure, which the two CLIs already do: it writes what they need, calls them, and records the job. The training script is the same file that runs locally; the wrapper installs sysone on the machine, sets `SYSONE_PRESET` to the machine's hardware preset (read by `sysone.learner`), runs the script (under `torchrun` on a multi-GPU machine), and pushes the export to the Hub so results survive the session.

Both CLIs upload a single script: Kaggle sends only the `code_file` named in its metadata, Colab sends one file as one cell. So the wrapper inlines the user's script and writes it back out on the machine. After the script, it looks for the export in `SYSONE_OUTPUT`, else for the newest folder holding a `sysone.json` (the script may call `learn.export("my-jev")` wherever it likes), tars it, and pushes it; a push that finds no export fails the job instead of passing quietly.

A job's life on Kaggle (Colab differs in the middle: `colab run` executes the file on a fresh VM, and `--keep` holds the VM for the download):

```{mermaid}
sequenceDiagram
    participant U as you
    participant S as sysone run
    participant K as Kaggle CLI
    participant G as the GPU session
    participant H as Hugging Face Hub
    U->>S: sysone run train.py --on kaggle --push me/jev
    S->>S: wrap_script: inline train.py, install sysone, set SYSONE_PRESET
    S->>K: kaggle kernels push: kernel-metadata.json, --accelerator
    K->>G: a session on the machine asked for
    G->>G: train.py trains and exports (under torchrun on two T4s)
    G->>H: the wrapper pushes the export
    S-->>U: the job, recorded in jobs.json
    U->>S: sysone jobs status
    S->>K: kaggle kernels status
    U->>S: fetch
    S->>K: kaggle kernels output: /kaggle/working and the export tarball
```

## The wrapper

In [ ]:
#| export
WRAPPER = '''\
# generated by sysone.cloud: installs sysone, runs the training script, pushes the export to the Hub
import glob, os, subprocess, sys, tarfile
os.chdir({workdir!r})
subprocess.run([sys.executable, "-m", "pip", "install", "-q", {install!r}], check=True)
os.environ.update({env!r})
os.environ.setdefault("SYSONE_OUTPUT", {output!r})
open("sysone_train.py", "w").write({script!r})
cmd = ([sys.executable, "-m", "torch.distributed.run", "--standalone", "--nproc_per_node", str({gpus})] if {gpus} > 1 else [sys.executable])
subprocess.run(cmd + ["sysone_train.py"] + {args!r}, check=True)
out = os.environ["SYSONE_OUTPUT"]
if not os.path.isdir(out):   # the export the script wrote, wherever it wrote it: the newest folder with sysone.json
    found = [os.path.dirname(p) for p in glob.glob("**/sysone.json", recursive=True) + glob.glob("**/sysone_gliner.json", recursive=True)]
    out = max(found, key=os.path.getmtime) if found else out
if not os.path.isdir(out) and {push!r}: sys.exit("sysone: the script wrote no export (no folder with sysone.json), so nothing was pushed")
if os.path.isdir(out):
    with tarfile.open("sysone_export.tar.gz", "w:gz") as t: t.add(out, arcname=os.path.basename(out))
    if {push!r}:
        token = os.environ.get("HF_TOKEN")
        if token is None:
            try:
                from kaggle_secrets import UserSecretsClient
                token = UserSecretsClient().get_secret("HF_TOKEN")
            except Exception: pass
        from huggingface_hub import HfApi
        api = HfApi(token=token)
        api.create_repo({push!r}, exist_ok=True, private=True)
        api.upload_folder(folder_path=out, repo_id={push!r}, commit_message="sysone export")
        print("pushed", out, "to", {push!r})
'''

def wrap_script(script:str|Path, workdir:str, preset:str, gpus:int=1, push:str|None=None, install:str="sysone",
                output:str|None=None, env:dict|None=None, args:list|None=None) -> str:
    "The single file a cloud machine runs: install sysone, run `script` with `SYSONE_PRESET`, tar and push the export"
    if isinstance(script, Path) or "\n" not in str(script):      # a path, not inline code: it must exist
        if not Path(str(script)).exists(): raise FileNotFoundError(f"no such training script: {script}")
        body = Path(script).read_text()
    else: body = str(script)
    env = {"SYSONE_PRESET": preset} | (env or {})
    return WRAPPER.format(workdir=workdir, install=install, env=env, output=output or os.path.join(workdir, "export"),
                          script=body, gpus=int(gpus), args=list(args or []), push=push)

In [ ]:
train_py = Path(tempfile.mkdtemp()) / "train.py"
train_py.write_text('from sysone.text import decision_learner, TypedDecisions\nprint("training")\n')
w = wrap_script(train_py, "/kaggle/working", "t4x2", gpus=2, push="user/my-jev", env={"SYSONE_DATA": "LocalLLaMA/typed-decisions"})
compile(w, "wrapper", "exec")
print(w[:600])
test_fail(lambda: wrap_script(Path("no/such/train.py"), "/w", "t4"), contains="no such training script")
test_fail(lambda: wrap_script("trian.py", "/w", "t4"), contains="no such training script")

# generated by sysone.cloud: installs sysone, runs the training script, pushes the export to the Hub
import glob, os, subprocess, sys, tarfile
os.chdir('/kaggle/working')
subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'sysone'], check=True)
os.environ.update({'SYSONE_PRESET': 't4x2', 'SYSONE_DATA': 'LocalLLaMA/typed-decisions'})
os.environ.setdefault("SYSONE_OUTPUT", '/kaggle/working/export')
open("sysone_train.py", "w").write('from sysone.text import decision_learner, TypedDecisions\nprint("training")\n')
cmd = ([sys.executable, "-m", "torch.distributed.run", "--standalone", "-


Run with its subprocesses stubbed, the wrapper finds an export the script wrote under a name of its own, tars it and pushes it, and fails when a push finds nothing:

In [ ]:
import sys, types, unittest.mock as mock
def run_wrapper(script_body, push="me/jev"):
    work = Path(tempfile.mkdtemp())
    code = wrap_script(script_body, str(work), "cpu", push=push)
    pushed = []
    api = types.SimpleNamespace(create_repo=lambda *a, **k: None, upload_folder=lambda **k: pushed.append(k["folder_path"]))
    def fake_run(cmd, check=False, **kw):      # pip install: skip; the training script: run it here
        if "sysone_train.py" in cmd: exec(compile(Path(work / "sysone_train.py").read_text(), "train", "exec"), {})
    with mock.patch("subprocess.run", fake_run), mock.patch.dict(sys.modules, {"huggingface_hub": types.SimpleNamespace(HfApi=lambda token=None: api)}):
        cwd = os.getcwd()
        try: exec(compile(code, "wrapper", "exec"), {})
        finally: os.chdir(cwd)
    return work, pushed

work, pushed = run_wrapper("import os, json\nos.makedirs('my-jev', exist_ok=True)\njson.dump({}, open('my-jev/sysone.json', 'w'))\n")
test_eq((len(pushed), Path(pushed[0]).name, (work / "sysone_export.tar.gz").exists()), (1, "my-jev", True))
test_fail(lambda: run_wrapper("print('trained, exported nothing')\n"), contains="wrote no export", exc=SystemExit)

pushed my-jev to me/jev
trained, exported nothing


In [ ]:
assert "'SYSONE_PRESET': 't4x2'" in w and "--nproc_per_node" in w and "user/my-jev" in w

Each machine type gets its preset: Kaggle's T4 accelerator is two T4s (`t4x2`, DDP through torchrun), an L4 is `l4`, an A100 or H100 is `a100`.

In [ ]:
#| export
KAGGLE_ACCELERATORS = {"NvidiaTeslaT4": ("t4x2", 2), "NvidiaL4": ("l4", 1), "NvidiaL4X1": ("l4", 1), "NvidiaTeslaA100": ("a100", 1),
                       "NvidiaH100": ("a100", 1), "NvidiaRtxPro6000": ("a100", 1)}
COLAB_GPUS = {"T4": ("t4", 1), "L4": ("l4", 1), "G4": ("l4", 1), "A100": ("a100", 1), "H100": ("a100", 1)}

## Jobs

A submitted job is recorded in `~/.config/sysone/jobs.json` (or `SYSONE_JOBS`), so `sysone jobs status` can list both backends' jobs and poll them; the files a job uploads are written under `~/.cache/sysone/jobs` (or `SYSONE_JOBS_DIR`), where they can be read before submitting.

In [ ]:
#| export
def jobs_file() -> Path: return Path(os.environ.get("SYSONE_JOBS", Path.home() / ".config" / "sysone" / "jobs.json"))
def jobs_dir() -> Path: return Path(os.environ.get("SYSONE_JOBS_DIR", Path.home() / ".cache" / "sysone" / "jobs"))

@dataclass
class Job:
    "A training run submitted to Kaggle or Colab"
    backend:str; ref:str; script:str; machine:str; push:str|None = None; submitted:float = field(default_factory=time.time)
    folder:str|None = None; status:str = "submitted"

def load_jobs() -> list:
    f = jobs_file()
    return [Job(**j) for j in json.loads(f.read_text())] if f.exists() else []

def save_job(job:Job) -> Job:
    jobs = [j for j in load_jobs() if (j.backend, j.ref) != (job.backend, job.ref)] + [job]
    jobs_file().parent.mkdir(parents=True, exist_ok=True)
    jobs_file().write_text(json.dumps([asdict(j) for j in jobs], indent=2))
    return job

In [ ]:
os.environ["SYSONE_JOBS"] = str(Path(tempfile.mkdtemp()) / "jobs.json")
os.environ["SYSONE_JOBS_DIR"] = tempfile.mkdtemp()
save_job(Job("kaggle", "me/sysone-train", "train.py", "NvidiaTeslaT4"))
save_job(Job("kaggle", "me/sysone-train", "train.py", "NvidiaTeslaT4", status="complete"))
test_eq([(j.ref, j.status) for j in load_jobs()], [("me/sysone-train", "complete")])

## Running a CLI

Every call goes through `_cli`, which finds the executable (installed with `sysone[cloud]`), runs it, and, with `dry_run=True`, only returns the command it would run. The Kaggle CLI reports some failures on stdout with exit code 0 (`Kernel push error: …`, a status of `ERROR`), so its output is parsed, not only its exit code.

In [ ]:
#| export
class CloudError(RuntimeError): "A cloud CLI reported a failure"

def _cli(cmd:list, dry_run:bool=False, check:bool=True):
    if dry_run: return {"cmd": cmd, "stdout": "", "returncode": 0}
    if shutil.which(cmd[0]) is None:
        raise CloudError(f"`{cmd[0]}` is not installed; pip install 'sysone[cloud]' (and authenticate: `kaggle auth login` or `colab --auth oauth2`)")
    r = subprocess.run(cmd, capture_output=True, text=True)
    out = {"cmd": cmd, "stdout": r.stdout, "stderr": r.stderr, "returncode": r.returncode}
    if check and (r.returncode != 0 or re.search(r"(push|creation) error", r.stdout, re.I)):
        raise CloudError(f"{shlex.join(cmd)} failed ({r.returncode}): {(r.stdout + r.stderr).strip()[-2000:]}")
    return out

In [ ]:
test_eq(_cli(["kaggle", "kernels", "list"], dry_run=True)["cmd"], ["kaggle", "kernels", "list"])

## Kaggle

`kaggle_kernel` writes the folder `kaggle kernels push` uploads: the wrapped script and `kernel-metadata.json` (a script kernel with internet on, the GPU given as `machine_shape`, Kaggle datasets or models to mount). `kaggle_run` pushes it with `--accelerator` and a session timeout (12 hours at most), `kaggle_status` reads the status line (old CLIs print `"complete"`, current ones `"KernelWorkerStatus.COMPLETE"`), and `kaggle_output` downloads `/kaggle/working`, the export tarball included.

In [ ]:
#| export
def kaggle_kernel(script, slug:str, folder=None, accelerator:str="NvidiaTeslaT4", push:str|None=None, install:str="sysone",
                  datasets:list|None=None, models:list|None=None, private:bool=True, env:dict|None=None, args:list|None=None) -> Path:
    "A folder for `kaggle kernels push`: the wrapped script and its kernel-metadata.json"
    if "/" not in slug: raise ValueError("the kernel slug is 'owner/name'")
    preset, gpus = KAGGLE_ACCELERATORS.get(accelerator, ("t4", 1))
    folder = Path(folder or jobs_dir() / "kaggle" / slug.split("/")[1]); folder.mkdir(parents=True, exist_ok=True)
    (folder / "sysone_job.py").write_text(wrap_script(script, "/kaggle/working", preset, gpus, push, install, env=env, args=args))
    name = slug.split("/")[1]
    meta = {"id": slug, "title": name.replace("-", " ").title() if len(name) >= 5 else f"sysone {name}", "code_file": "sysone_job.py",
            "language": "python", "kernel_type": "script", "is_private": private, "enable_gpu": True, "enable_internet": True,
            "machine_shape": accelerator, "dataset_sources": list(datasets or []), "competition_sources": [],
            "kernel_sources": [], "model_sources": list(models or [])}
    (folder / "kernel-metadata.json").write_text(json.dumps(meta, indent=2))
    return folder

def kaggle_run(folder, accelerator:str="NvidiaTeslaT4", timeout:int=12 * 3600, dry_run:bool=False) -> Job:
    "Push a kernel folder and record the job"
    folder = Path(folder); meta = json.loads((folder / "kernel-metadata.json").read_text())
    _cli(["kaggle", "kernels", "push", "-p", str(folder), "--accelerator", accelerator, "-t", str(int(timeout))], dry_run)
    job = Job("kaggle", meta["id"], str(folder / "sysone_job.py"), accelerator, folder=str(folder))
    return job if dry_run else save_job(job)

def parse_kaggle_status(text:str) -> str:
    "The status word of `kaggle kernels status` output: queued, running, complete, error, cancelacknowledged"
    m = re.search(r'has status "([^"]+)"', text)
    if not m: raise CloudError(f"unrecognised status output: {text[:200]!r}")
    return m.group(1).split(".")[-1].replace("_", "").lower()

def kaggle_status(ref:str, dry_run:bool=False) -> str:
    "The kernel's status"
    out = _cli(["kaggle", "kernels", "status", ref], dry_run)
    return "dry-run" if dry_run else parse_kaggle_status(out["stdout"])

def kaggle_output(ref:str, dest, dry_run:bool=False) -> Path:
    "Download the kernel's outputs (the export tarball among them) to `dest`"
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    _cli(["kaggle", "kernels", "output", ref, "-p", str(dest)], dry_run)
    return dest

In [ ]:
kf = kaggle_kernel(train_py, "me/sysone-typed-decisions", Path(tempfile.mkdtemp()) / "k", push="me/laya-td")
meta = json.loads((kf / "kernel-metadata.json").read_text())
meta

{'id': 'me/sysone-typed-decisions',
 'title': 'Sysone Typed Decisions',
 'code_file': 'sysone_job.py',
 'language': 'python',
 'kernel_type': 'script',
 'is_private': True,
 'enable_gpu': True,
 'enable_internet': True,
 'machine_shape': 'NvidiaTeslaT4',
 'dataset_sources': [],
 'competition_sources': [],
 'kernel_sources': [],
 'model_sources': []}

In [ ]:
test_eq((meta["code_file"], meta["kernel_type"], meta["machine_shape"], meta["enable_internet"]), ("sysone_job.py", "script", "NvidiaTeslaT4", True))
job = kaggle_run(kf, dry_run=True)
test_eq(job.ref, "me/sysone-typed-decisions")
test_eq(parse_kaggle_status('me/k has status "KernelWorkerStatus.COMPLETE"'), "complete")
test_eq(parse_kaggle_status('me/k has status "cancelAcknowledged"'), "cancelacknowledged")
test_fail(lambda: kaggle_kernel(train_py, "no-owner"), contains="owner/name")

## Colab

`colab run` executes one file on a fresh VM and releases it; `--timeout` is not a run limit, so it is set above the job's length (a low value makes the CLI busy-wait and can fail a finished run), and `--keep` keeps the VM so the export tarball can be downloaded with `colab download` (single files only, which is why the wrapper tars the export) before `colab stop`. For long jobs the wrapper's Hub push is the safer way back: a dropped connection ends the session.

In [ ]:
#| export
def colab_run(script, gpu:str="L4", session:str|None=None, push:str|None=None, keep:bool=True, install:str="sysone",
              timeout:int=24 * 3600, env:dict|None=None, args:list|None=None, dry_run:bool=False, folder=None) -> Job:
    "Run a training script on a Colab GPU VM (kept for downloading the export unless keep=False)"
    preset, gpus = COLAB_GPUS.get(gpu.upper(), ("t4", 1))
    session = session or f"sysone-{uuid.uuid4().hex[:6]}"
    folder = Path(folder or jobs_dir() / "colab" / session); folder.mkdir(parents=True, exist_ok=True)
    wrapped = folder / "sysone_job.py"
    wrapped.write_text(wrap_script(script, "/content", preset, gpus, push, install, env=env, args=args))
    cmd = ["colab", "run", "--gpu", gpu.upper(), "-s", session, "--timeout", str(int(timeout))] + (["--keep"] if keep else []) + [str(wrapped)]
    job = Job("colab", session, str(wrapped), gpu.upper(), push=push, folder=str(folder))
    if not dry_run: save_job(job)
    _cli(cmd, dry_run)
    if not dry_run: job.status = "complete"; save_job(job)
    return job

def colab_download(session:str, dest, dry_run:bool=False) -> Path:
    "Download a kept session's export tarball"
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    _cli(["colab", "download", "-s", session, "/content/sysone_export.tar.gz", str(dest / "sysone_export.tar.gz")], dry_run)
    return dest

def colab_stop(session:str, dry_run:bool=False): return _cli(["colab", "stop", "-s", session], dry_run)

In [ ]:
cj = colab_run(train_py, gpu="l4", session="sysone-test", dry_run=True, folder=Path(tempfile.mkdtemp()))
test_eq((cj.backend, cj.machine), ("colab", "L4"))
assert "'SYSONE_PRESET': 'l4'" in Path(cj.script).read_text()

## run

`run` is what `sysone run` calls: one script, one backend, one machine. `status` refreshes and returns the recorded jobs, and `fetch` gets a finished job's export.

In [ ]:
#| export
def run(script, on:str="kaggle", accelerator:str|None=None, gpu:str|None=None, slug:str|None=None, push:str|None=None,
        data:str|None=None, keep:bool=True, install:str="sysone", dry_run:bool=False, **kw) -> Job:
    "Submit a training script to Kaggle or Colab"
    env = {"SYSONE_DATA": data} if data else {}
    if on == "kaggle":
        if slug is None:
            user = os.environ.get("KAGGLE_USERNAME")
            if not user: raise ValueError("name the kernel with slug='owner/name' (or set KAGGLE_USERNAME)")
            slug = f"{user}/sysone-{Path(str(script)).stem.replace('_', '-')}"
        acc = accelerator or "NvidiaTeslaT4"
        datasets = [data[len("kaggle:"):]] if data and data.startswith("kaggle:") else None
        folder = kaggle_kernel(script, slug, kw.pop("folder", None), acc, push, install, datasets=datasets, env=env, **kw)
        return kaggle_run(folder, acc, dry_run=dry_run)
    if on == "colab": return colab_run(script, gpu or "L4", push=push, keep=keep, install=install, env=env, dry_run=dry_run, **kw)
    raise ValueError(f"unknown backend {on!r}; use 'kaggle' or 'colab'")

def status(refresh:bool=True) -> list:
    "The recorded jobs, with Kaggle statuses refreshed"
    jobs = load_jobs()
    for j in jobs:
        if refresh and j.backend == "kaggle" and j.status not in ("complete", "error", "cancelacknowledged"):
            try: j.status = kaggle_status(j.ref); save_job(j)
            except CloudError as e: j.status = f"unknown ({str(e)[:60]})"
    return jobs

def fetch(job:Job, dest) -> Path:
    "Download a finished job's export tarball and unpack it"
    import tarfile
    dest = Path(dest)
    (kaggle_output(job.ref, dest) if job.backend == "kaggle" else colab_download(job.ref, dest))
    tb = dest / "sysone_export.tar.gz"
    if tb.exists():
        with tarfile.open(tb) as t: t.extractall(dest, filter="data")
    return dest

In [ ]:
j = run(train_py, on="kaggle", slug="me/sysone-demo", push="me/demo-jev", data="LocalLLaMA/typed-decisions", dry_run=True,
        folder=Path(tempfile.mkdtemp()))
test_eq(j.machine, "NvidiaTeslaT4")
test_fail(lambda: run(train_py, on="lambda"), contains="unknown backend")

## Publishing to Kaggle Models

The Hub is where exports go by default; `publish_kaggle` also makes a Kaggle Model from an export folder, writing the two metadata files the CLI reads and uploading the folder as a variation (its subfolders zipped, which the CLI otherwise skips).

In [ ]:
#| export
def publish_kaggle(path, owner:str, model:str, variation:str="default", license:str="Apache 2.0", framework:str="pyTorch",
                   title:str|None=None, private:bool=True, dry_run:bool=False) -> dict:
    "Create a Kaggle Model and upload an export folder as one of its variations"
    path = Path(path)
    meta_dir = Path(tempfile.mkdtemp(prefix="sysone-kaggle-model-"))
    (meta_dir / "model-metadata.json").write_text(json.dumps({"ownerSlug": owner, "title": title or model, "slug": model,
        "subtitle": "", "isPrivate": private, "description": f"# {title or model}\n\nA typed-decision model exported by sysone.\n",
        "publishTime": "", "provenanceSources": ""}, indent=2))
    (path / "model-instance-metadata.json").write_text(json.dumps({"ownerSlug": owner, "modelSlug": model, "instanceSlug": variation,
        "framework": framework, "overview": "sysone export: sysone.json, head.safetensors, tokenizer/, encoder/",
        "usage": "from sysone.inference import Decider\nDecider.load(path).predict(state, questions)", "licenseName": license,
        "fineTunable": True, "trainingData": [], "modelInstanceType": "Unspecified", "baseModelInstance": "", "externalBaseModelUrl": ""}, indent=2))
    a = _cli(["kaggle", "models", "create", "-p", str(meta_dir)], dry_run)
    b = _cli(["kaggle", "models", "variations", "create", "-p", str(path), "-r", "zip"], dry_run)
    return {"model": a["cmd"], "variation": b["cmd"]}

In [ ]:
exp = Path(tempfile.mkdtemp()); (exp / "sysone.json").write_text("{}")
cmds = publish_kaggle(exp, "me", "my-jev", dry_run=True)
test_eq(cmds["variation"][-2:], ["-r", "zip"])
test_eq(json.loads((exp / "model-instance-metadata.json").read_text())["instanceSlug"], "default")

Out of scope on purpose: a job queue, cost tracking, notebooks as the unit of work (scripts only, which both CLIs run), and any provider without a CLI. The real runs need credentials (`kaggle auth login`, or `KAGGLE_API_TOKEN`; `colab --auth oauth2` once, interactively) and an `HF_TOKEN` secret for the push; with them, the reproduction of Laya's fine-tune in the text notebook is:

In [ ]:
#| eval: false
job = run("reproduce_laya.py", on="kaggle", accelerator="NvidiaTeslaT4", slug="me/sysone-laya-td", push="me/laya-typed-decisions")
status()
fetch(job, "laya-typed-decisions")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()